In [3]:
import sys
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec
from matplotlib.backends.backend_pdf import PdfPages

ROOT = Path("/Users/andreali/Documents/Subgraph_Federated_Learning/")

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from andrea.multigraph_generation import TASKS




In [10]:
SELECT_SUBSET_PATH = "clustering"
SELECT_SUBSET = "selected_subset"

EXPERIMENT_LOG_FOLDER = "experiment_log.csv"
DATA_DIR = "clustering/cluster_generation_parameters.csv"

SELECTED_SUBSETS_CSV_PATH = Path(f"./{SELECT_SUBSET_PATH}/{SELECT_SUBSET}.csv")
selected_subset = pd.read_csv(SELECTED_SUBSETS_CSV_PATH)
print("Loaded selected pairs rows:", len(selected_subset))

exp_log = pd.read_csv(EXPERIMENT_LOG_FOLDER)
print("exp rows:", len(exp_log))

test_gen = pd.read_csv(DATA_DIR)
print("Loaded generation rows:", len(test_gen))

Loaded selected pairs rows: 1
exp rows: 63
Loaded generation rows: 170


In [30]:
print(exp_log["model_tag"].tolist())

['mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64', 'mcwauto_layers6_lr0.001_wd0.0001_do0.1

## 1. Build plotting tables

In [42]:
def parse_subset_clients_str(s):
    return [str(x).strip() for x in str(s).split("|") if str(x).strip() != ""]

def parse_json_list_safe(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, list):
        return x
    return json.loads(x)

def parse_json_dict_safe(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {}
    if isinstance(x, dict):
        return x
    return json.loads(x)

def _available_model_tags(local_match, fed_match):
    return sorted(set(local_match["model_tag"]).intersection(set(fed_match["model_tag"])))

def _available_seeds(local_match, fed_match):
    local_seeds = set(pd.to_numeric(local_match["seed"], errors="coerce").dropna().astype(int))
    fed_seeds = set(pd.to_numeric(fed_match["seed"], errors="coerce").dropna().astype(int))
    return sorted(local_seeds.intersection(fed_seeds))


def build_global_run_table(selected_subset_df, exp_log):
    rows = []

    local_exp_log = exp_log[exp_log["run_type"] == "local"].copy()
    fedavg_exp_log = exp_log[exp_log["run_type"] == "fedavg"].copy()

    for _, sel_row in selected_subset_df.iterrows():
        subset_clients = str(sel_row["subset_clients"])
        graph_ids = [str(x) for x in parse_json_list_safe(sel_row["graph_ids_json"])]
        family_order = parse_json_list_safe(sel_row["family_order_json"])
        family_to_graph_ids = {
            str(k): [str(v) for v in vals]
            for k, vals in parse_json_dict_safe(sel_row["family_to_graph_ids_json"]).items()
        }
        family_counts = parse_json_dict_safe(sel_row["family_counts_json"])

        local_match = local_exp_log[local_exp_log["subset_clients"] == subset_clients].copy()
        fed_match = fedavg_exp_log[fedavg_exp_log["subset_clients"] == subset_clients].copy()

        for model_tag in _available_model_tags(local_match, fed_match):
            local_model = local_match[local_match["model_tag"] == model_tag].copy()
            fed_model = fed_match[fed_match["model_tag"] == model_tag].copy()
            seeds = _available_seeds(local_model, fed_model)

            if not seeds:
                continue

            rows.append(
                {
                    "subset_id": sel_row.get("subset_id", None),
                    "subset_clients": subset_clients,
                    "subset_size": int(sel_row.get("subset_size", len(graph_ids))),
                    "graph_ids": graph_ids,
                    "family_order": family_order,
                    "family_to_graph_ids": family_to_graph_ids,
                    "family_counts": family_counts,
                    "model_tag": model_tag,
                    "local_epochs": int(fed_model["local_epochs"].iloc[0]),
                    "seeds": seeds,
                }
            )

    return pd.DataFrame(rows)


def build_family_run_table(selected_subset_df, exp_log):
    rows = []

    local_exp_log = exp_log[exp_log["run_type"] == "local"].copy()
    fedavg_exp_log = exp_log[exp_log["run_type"] == "fedavg"].copy()

    for _, sel_row in selected_subset_df.iterrows():
        subset_clients = str(sel_row["subset_clients"])
        family_order = parse_json_list_safe(sel_row["family_order_json"])
        family_to_graph_ids = {
            str(k): [str(v) for v in vals]
            for k, vals in parse_json_dict_safe(sel_row["family_to_graph_ids_json"]).items()
        }

        local_match = local_exp_log[local_exp_log["subset_clients"] == subset_clients].copy()
        fed_match = fedavg_exp_log[fedavg_exp_log["subset_clients"] == subset_clients].copy()

        for model_tag in _available_model_tags(local_match, fed_match):
            local_model = local_match[local_match["model_tag"] == model_tag].copy()
            fed_model = fed_match[fed_match["model_tag"] == model_tag].copy()
            seeds = _available_seeds(local_model, fed_model)

            if not seeds:
                continue

            ordered_families = family_order if len(family_order) > 0 else list(family_to_graph_ids.keys())

            for family in ordered_families:
                graph_ids = family_to_graph_ids.get(str(family), [])
                if not graph_ids:
                    continue

                rows.append(
                    {
                        "subset_id": sel_row.get("subset_id", None),
                        "subset_clients": subset_clients,
                        "family": str(family),
                        "subset_size": len(graph_ids),
                        "graph_ids": [str(g) for g in graph_ids],
                        "model_tag": model_tag,
                        "local_epochs": int(fed_model["local_epochs"].iloc[0]),
                        "seeds": seeds,
                    }
                )

    return pd.DataFrame(rows)


def build_client_run_table(selected_subset_df, exp_log):
    rows = []

    local_exp_log = exp_log[exp_log["run_type"] == "local"].copy()
    fedavg_exp_log = exp_log[exp_log["run_type"] == "fedavg"].copy()

    for _, sel_row in selected_subset_df.iterrows():
        subset_clients = str(sel_row["subset_clients"])
        graph_to_family = {
            str(k): str(v)
            for k, v in parse_json_dict_safe(sel_row["graph_to_family_json"]).items()
        }
        graph_ids = [str(x) for x in parse_json_list_safe(sel_row["graph_ids_json"])]

        local_match = local_exp_log[local_exp_log["subset_clients"] == subset_clients].copy()
        fed_match = fedavg_exp_log[fedavg_exp_log["subset_clients"] == subset_clients].copy()

        for model_tag in _available_model_tags(local_match, fed_match):
            local_model = local_match[local_match["model_tag"] == model_tag].copy()
            fed_model = fed_match[fed_match["model_tag"] == model_tag].copy()
            seeds = _available_seeds(local_model, fed_model)

            if not seeds:
                continue

            for graph_id in graph_ids:
                rows.append(
                    {
                        "subset_id": sel_row.get("subset_id", None),
                        "subset_clients": subset_clients,
                        "family": graph_to_family.get(str(graph_id), "unknown"),
                        "graph_id": str(graph_id),
                        "model_tag": model_tag,
                        "local_epochs": int(fed_model["local_epochs"].iloc[0]),
                        "seeds": seeds,
                    }
                )

    return pd.DataFrame(rows)

## General helpers

In [59]:
def resolve_existing_path(path_like):
    p = Path(path_like)
    candidates = [
        p,
        Path.cwd() / p,
        ROOT / p,
    ]
    for cand in candidates:
        if cand.exists():
            return cand.resolve()
    raise FileNotFoundError(f"Could not resolve path: {path_like}")

def load_run_csv(csv_path_like) -> pd.DataFrame:
    csv_path = resolve_existing_path(csv_path_like)
    return pd.read_csv(csv_path)

## Data collection for plotting


In [49]:
def aggregate_seed_curves(curves, value_col):
    parts = []
    for seed_idx, curve in enumerate(curves):
        if curve is None or curve.empty:
            continue
        part = curve[["step", value_col]].dropna().copy()
        if part.empty:
            continue
        part["seed_idx"] = seed_idx
        parts.append(part)

    if not parts:
        return pd.DataFrame(columns=["step", "mean", "std", "count"])

    full = pd.concat(parts, axis=0, ignore_index=True)
    agg = (
        full.groupby("step")[value_col]
        .agg(["mean", "std", "count"])
        .reset_index()
        .sort_values("step")
    )
    agg["std"] = agg["std"].fillna(0.0)
    return agg


def get_scalar_seed_stats(
    dfs,
    *,
    phase,
    split,
    metric_col,
    graph_id=None,
    local_epochs=1,
):
    curves = []
    for df in dfs:
        part = get_scalar_curve(
            df,
            phase=phase,
            split=split,
            metric_col=metric_col,
            graph_id=graph_id,
            local_epochs=local_epochs,
        )
        curves.append(part)
    return aggregate_seed_curves(curves, metric_col)


def get_task_seed_stats(
    dfs,
    *,
    phase,
    split,
    task,
    metric_col,
    graph_id=None,
    local_epochs=1,
):
    curves = []
    for df in dfs:
        part = get_task_curve(
            df,
            phase=phase,
            split=split,
            task=task,
            metric_col=metric_col,
            graph_id=graph_id,
            local_epochs=local_epochs,
        )
        curves.append(part)
    return aggregate_seed_curves(curves, metric_col)


def get_delta_seed_stats(
    left_dfs,
    right_dfs,
    *,
    phase_left,
    phase_right,
    split,
    metric_col,
    graph_id_right=None,
    task=None,
    local_epochs=1,
):
    delta_curves = []

    for left_df, right_df in zip(left_dfs, right_dfs):
        if task is None:
            left = get_scalar_curve(
                left_df,
                phase=phase_left,
                split=split,
                metric_col=metric_col,
                local_epochs=local_epochs,
            )
            right = get_scalar_curve(
                right_df,
                phase=phase_right,
                split=split,
                metric_col=metric_col,
                graph_id=graph_id_right,
                local_epochs=local_epochs,
            )
        else:
            left = get_task_curve(
                left_df,
                phase=phase_left,
                split=split,
                task=task,
                metric_col=metric_col,
                local_epochs=local_epochs,
            )
            right = get_task_curve(
                right_df,
                phase=phase_right,
                split=split,
                task=task,
                metric_col=metric_col,
                graph_id=graph_id_right,
                local_epochs=local_epochs,
            )

        if left.empty or right.empty:
            continue

        merged = pd.merge(
            left[["step", metric_col]],
            right[["step", metric_col]],
            on="step",
            how="inner",
            suffixes=("_left", "_right"),
        )
        if merged.empty:
            continue

        merged["delta"] = merged[f"{metric_col}_left"] - merged[f"{metric_col}_right"]
        delta_curves.append(merged[["step", "delta"]])

    return aggregate_seed_curves(delta_curves, "delta")


def plot_mean_std(
    ax,
    agg_df,
    *,
    label,
    linestyle="-",
    color=None,
    alpha_fill=0.16,
    linewidth=2.0,
):
    if agg_df is None or agg_df.empty:
        return False

    x = agg_df["step"].to_numpy()
    y = agg_df["mean"].to_numpy()
    s = agg_df["std"].to_numpy()

    line, = ax.plot(
        x, y,
        label=label,
        linestyle=linestyle,
        color=color,
        linewidth=linewidth,
    )
    fill_color = line.get_color()
    ax.fill_between(x, y - s, y + s, alpha=alpha_fill, color=fill_color)
    return True

def _effective_step(df, local_epochs):
    df = df.copy()
    if df.empty:
        return df

    phase = None
    if "phase" in df.columns and df["phase"].notna().any():
        phase = str(df["phase"].dropna().iloc[0])

    has_round = "round" in df.columns and df["round"].notna().any()
    has_local_epoch = "local_epoch" in df.columns and df["local_epoch"].notna().any()

    ROUND_BASED_PHASES = {
        "val_epoch",                 # fedavg pre-aggregation evals
        "val_epoch_task",            # fedavg pre-aggregation per-task evals
        "global_val_client",
        "global_val_client_task",
        "global_val_mean",
        "best_global_train",
        "best_global_train_task",
        "best_global_val",
        "best_global_val_task",
        "best_global_test",
        "best_global_test_task",
    }

    # FedAvg eval phases: one point per round
    if phase in ROUND_BASED_PHASES and has_round:
        df["step"] = df["round"].astype(int)

    # phases that truly have both round and local_epoch (e.g. train_epoch in fedavg)
    elif has_round and has_local_epoch:
        df["step"] = (
            (df["round"].astype(int) - 1) * int(local_epochs)
            + df["local_epoch"].astype(int)
        )

    # local standalone phases
    elif has_local_epoch:
        df["step"] = df["local_epoch"].astype(int)

    # fallback
    elif has_round:
        df["step"] = df["round"].astype(int)

    else:
        raise ValueError("Could not infer step from dataframe")

    return df

def get_scalar_curve(df, *, phase, split, metric_col, graph_id=None, local_epochs=1):
    part = df[(df["phase"] == phase) & (df["split"] == split) & (df["task"].isna())].copy()
    if graph_id is not None:
        part = part[part["graph_id"].astype(str) == str(graph_id)].copy()
    if part.empty:
        return part

    part = _effective_step(part, local_epochs)
    return part[["step", metric_col]].dropna().sort_values("step")

def get_task_curve(df, *, phase, split, task, metric_col, graph_id=None, local_epochs=1):
    part = df[
        (df["phase"] == phase)
        & (df["split"] == split)
        & (df["task"] == task)
    ].copy()

    if graph_id is not None:
        part = part[part["graph_id"].astype(str) == str(graph_id)].copy()
    if part.empty:
        return part

    part = _effective_step(part, local_epochs)
    return part[["step", metric_col]].dropna().sort_values("step")

def _empty_curve(value_name="value"):
    return pd.DataFrame(columns=["step", value_name])


def get_task_curve_named(
    df,
    *,
    phase,
    split,
    task,
    metric_col,
    value_name,
    graph_id=None,
    local_epochs=1,
):
    part = get_task_curve(
        df,
        phase=phase,
        split=split,
        task=task,
        metric_col=metric_col,
        graph_id=graph_id,
        local_epochs=local_epochs,
    )
    if part.empty:
        return _empty_curve(value_name)
    return part.rename(columns={metric_col: value_name})


def merge_two_curves(left_df, right_df, left_name, right_name, delta_name):
    if left_df.empty or right_df.empty:
        return pd.DataFrame(columns=["step", left_name, right_name, delta_name])

    merged = pd.merge(left_df, right_df, on="step", how="inner")
    if merged.empty:
        return pd.DataFrame(columns=["step", left_name, right_name, delta_name])

    merged[delta_name] = merged[left_name] - merged[right_name]
    return merged.sort_values("step").reset_index(drop=True)


def annotate_no_data(ax, text="No data"):
    ax.text(
        0.5,
        0.5,
        text,
        ha="center",
        va="center",
        transform=ax.transAxes,
        fontsize=10,
        color="gray",
    )

def fmt_percent(rate):
    if rate is None or pd.isna(rate):
        return "NA"
    return f"{100.0 * float(rate):.1f}%"

def fmt_count(x):
    if x is None or pd.isna(x):
        return "NA"
    return f"{int(round(float(x)))}"

def fmt_rate_count(rate, count):
    return f"rate={fmt_percent(rate)}, count={fmt_count(count)}"

def _soft_cell_color(value, *, cmap_name="Blues", vmin=0.0, vmax=0.7, alpha=0.35):
    """
    value should be in raw rate space, e.g. 0.249 not 24.9
    """
    if value is None or pd.isna(value):
        return (1, 1, 1, 1)

    value = float(value)
    norm = (value - vmin) / max(vmax - vmin, 1e-12)
    norm = min(max(norm, 0.0), 1.0)

    cmap = plt.get_cmap(cmap_name)
    rgba = cmap(0.18 + 0.55 * norm)
    return (rgba[0], rgba[1], rgba[2], alpha)



def plot_task_family_panel(
    ax,
    df,
    *,
    phase,
    split,
    graph_id,
    local_epochs,
    metric_col,
    title,
):
    has_any = False
    for task in TASKS:
        curve = get_task_curve_named(
            df,
            phase=phase,
            split=split,
            task=task,
            metric_col=metric_col,
            value_name=metric_col,
            graph_id=graph_id,
            local_epochs=local_epochs,
        )
        if curve.empty:
            continue
        has_any = True
        ax.plot(curve["step"], curve[metric_col], label=task)

    ax.set_title(title)
    ax.set_xlabel("communication round / local epoch")
    ax.set_ylabel(metric_col)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)

    if not has_any:
        annotate_no_data(ax, f"No valid {metric_col} points")
    else:
        ax.legend(fontsize=8, ncol=2)


def plot_task_delta_panel(
    ax,
    local_df,
    fed_df,
    *,
    graph_id,
    local_epochs,
    metric_col,
    title,
):
    has_any = False
    for task in TASKS:
        local_curve = get_task_curve_named(
            local_df,
            phase="val_epoch_task",
            split="val",
            task=task,
            metric_col=metric_col,
            value_name="local_val",
            local_epochs=local_epochs,
        )
        global_curve = get_task_curve_named(
            fed_df,
            phase="global_val_client_task",
            split="val",
            task=task,
            metric_col=metric_col,
            value_name="global_val",
            graph_id=graph_id,
            local_epochs=local_epochs,
        )
        merged = merge_two_curves(
            local_curve,
            global_curve,
            "local_val",
            "global_val",
            "delta",
        )
        if merged.empty:
            continue
        has_any = True
        ax.plot(merged["step"], merged["delta"], label=task)

    ax.axhline(0.0, linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("communication round / local epoch")
    ax.set_ylabel(f"local - global {metric_col}")
    ax.grid(alpha=0.3)

    if not has_any:
        annotate_no_data(ax, f"No valid delta for {metric_col}")
    else:
        ax.legend(fontsize=8, ncol=2)

def get_task_count_curve(
    df,
    *,
    phase,
    split,
    task,
    graph_id=None,
    local_epochs=1,
):
    part = df[
        (df["phase"] == phase)
        & (df["split"] == split)
        & (df["task"] == task)
    ].copy()

    if graph_id is not None:
        part = part[part["graph_id"].astype(str) == str(graph_id)].copy()

    needed = ["tp", "fp", "tn", "fn"]
    if part.empty or any(c not in part.columns for c in needed):
        return pd.DataFrame(columns=["step", "tp", "fp", "tn", "fn"])

    part = _effective_step(part, local_epochs)
    part = part[["step", "tp", "fp", "tn", "fn"]].dropna()
    return part.sort_values("step").reset_index(drop=True)


def binary_f1_from_counts(tp, fp, fn):
    denom = 2.0 * tp + fp + fn
    out = np.full_like(tp, np.nan, dtype=float)
    valid = denom > 0
    out[valid] = (2.0 * tp[valid]) / denom[valid]
    return out


## 2. Filter experiment rows and load run CSVs


In [61]:
def filter_local_manifest(exp_log, subset_clients, model_tag, graph_ids=None):
    part = exp_log[exp_log["run_type"] == "local"].copy()
    part = part[
        (part["subset_clients"] == str(subset_clients)) &
        (part["model_tag"] == model_tag)
    ].copy()

    if graph_ids is not None:
        graph_ids = {str(g) for g in graph_ids}
        part = part[part["graph_id"].astype(str).isin(graph_ids)].copy()

    part["seed"] = pd.to_numeric(part["seed"], errors="coerce").astype("Int64")
    return part.sort_values(["seed", "graph_id"]).reset_index(drop=True)


def filter_fedavg_manifest(exp_log, subset_clients, model_tag):
    part = exp_log[exp_log["run_type"] == "fedavg"].copy()
    part = part[
        (part["subset_clients"] == str(subset_clients)) &
        (part["model_tag"] == model_tag)
    ].copy()

    part["seed"] = pd.to_numeric(part["seed"], errors="coerce").astype("Int64")
    return part.sort_values(["seed"]).reset_index(drop=True)


def load_local_items(local_rows):
    items = []
    for _, row in local_rows.iterrows():
        items.append(
            {
                "seed": int(row["seed"]),
                "graph_id": str(row["graph_id"]),
                "family": row.get("family", None),
                "local_epochs": int(row["local_epochs"]),
                "df": load_run_csv(row["out_csv"]),
            }
        )
    return items


def load_fedavg_items(fed_rows):
    items = []
    for _, row in fed_rows.iterrows():
        items.append(
            {
                "seed": int(row["seed"]),
                "local_epochs": int(row["local_epochs"]),
                "df": load_run_csv(row["out_csv"]),
            }
        )
    return items


def build_global_title(row):
    return (
        f"Aggregated {int(row['subset_size'])}-clients comparison with FedAvg"
        f" | subset_size={int(row['subset_size'])}"
        f" | model={row['model_tag']}"
        f" | local_epochs={int(row['local_epochs'])}"
        f" | n_seeds={len(row['seeds'])}"
    )


def build_family_title(row):
    return (
        f"family={row['family']}"
        f" | subset_size={int(row['subset_size'])}"
        f" | model={row['model_tag']}"
        f" | local_epochs={int(row['local_epochs'])}"
        f" | n_seeds={len(row['seeds'])}"
    )


def build_client_title(row):
    return (
        f"family={row['family']}"
        f" | graph_id={row['graph_id']}"
        f" | model={row['model_tag']}"
        f" | local_epochs={int(row['local_epochs'])}"
        f" | n_seeds={len(row['seeds'])}"
    )

## 3. Aggregate curves at group level


In [62]:
def get_group_scalar_stats(
    *,
    items,
    phase,
    split,
    metric_col,
    graph_ids=None,
    use_graph_filter=True,
):
    curves = []

    for item in items:
        df = item["df"]
        local_epochs = int(item["local_epochs"])

        if graph_ids is None or not use_graph_filter:
            curve = get_scalar_curve(
                df,
                phase=phase,
                split=split,
                metric_col=metric_col,
                graph_id=None,
                local_epochs=local_epochs,
            )
            curves.append(curve)
        else:
            for gid in graph_ids:
                curve = get_scalar_curve(
                    df,
                    phase=phase,
                    split=split,
                    metric_col=metric_col,
                    graph_id=str(gid),
                    local_epochs=local_epochs,
                )
                curves.append(curve)

    return aggregate_seed_curves(curves, metric_col)


def get_group_task_stats(
    *,
    items,
    phase,
    split,
    task,
    metric_col,
    graph_ids=None,
    use_graph_filter=True,
):
    curves = []

    for item in items:
        df = item["df"]
        local_epochs = int(item["local_epochs"])

        if graph_ids is None or not use_graph_filter:
            curve = get_task_curve(
                df,
                phase=phase,
                split=split,
                task=task,
                metric_col=metric_col,
                graph_id=None,
                local_epochs=local_epochs,
            )
            curves.append(curve)
        else:
            for gid in graph_ids:
                curve = get_task_curve(
                    df,
                    phase=phase,
                    split=split,
                    task=task,
                    metric_col=metric_col,
                    graph_id=str(gid),
                    local_epochs=local_epochs,
                )
                curves.append(curve)

    return aggregate_seed_curves(curves, metric_col)


def get_group_delta_stats(
    *,
    local_items,
    fedavg_items,
    split,
    metric_col,
    graph_ids,
    task=None,
):
    fed_by_seed = {int(item["seed"]): item for item in fedavg_items}
    delta_curves = []

    target_graph_ids = {str(g) for g in graph_ids}

    for item in local_items:
        seed = int(item["seed"])
        graph_id = str(item["graph_id"])

        if graph_id not in target_graph_ids:
            continue
        if seed not in fed_by_seed:
            continue

        local_df = item["df"]
        fed_df = fed_by_seed[seed]["df"]
        local_epochs = int(item["local_epochs"])

        if task is None:
            left = get_scalar_curve(
                local_df,
                phase="val_epoch",
                split=split,
                metric_col=metric_col,
                graph_id=None,
                local_epochs=local_epochs,
            )
            right = get_scalar_curve(
                fed_df,
                phase="global_val_client",
                split=split,
                metric_col=metric_col,
                graph_id=graph_id,
                local_epochs=local_epochs,
            )
        else:
            left = get_task_curve(
                local_df,
                phase="val_epoch_task",
                split=split,
                task=task,
                metric_col=metric_col,
                graph_id=None,
                local_epochs=local_epochs,
            )
            right = get_task_curve(
                fed_df,
                phase="global_val_client_task",
                split=split,
                task=task,
                metric_col=metric_col,
                graph_id=graph_id,
                local_epochs=local_epochs,
            )

        if left.empty or right.empty:
            continue

        merged = pd.merge(
            left[["step", metric_col]],
            right[["step", metric_col]],
            on="step",
            how="inner",
            suffixes=("_local", "_fed"),
        )
        if merged.empty:
            continue

        merged["delta"] = merged[f"{metric_col}_local"] - merged[f"{metric_col}_fed"]
        delta_curves.append(merged[["step", "delta"]])

    return aggregate_seed_curves(delta_curves, "delta")


def _pool_count_curves(curves):
    good = []
    for curve in curves:
        if curve is None or curve.empty:
            continue
        good.append(curve[["step", "tp", "fp", "tn", "fn"]].copy())

    if not good:
        return pd.DataFrame(columns=["step", "tp", "fp", "tn", "fn"])

    full = pd.concat(good, axis=0, ignore_index=True)
    pooled = (
        full.groupby("step")[["tp", "fp", "tn", "fn"]]
        .sum()
        .reset_index()
        .sort_values("step")
    )
    return pooled


def get_group_pooled_task_f1_local(
    *,
    local_items,
    task,
    graph_ids,
    split="val",
):
    target_graph_ids = {str(g) for g in graph_ids}
    curves = []

    seeds = sorted({int(item["seed"]) for item in local_items})

    for seed in seeds:
        count_curves = []
        seed_items = [
            item for item in local_items
            if int(item["seed"]) == seed and str(item["graph_id"]) in target_graph_ids
        ]

        for item in seed_items:
            count_curve = get_task_count_curve(
                item["df"],
                phase="val_epoch_task",
                split=split,
                task=task,
                graph_id=None,
                local_epochs=int(item["local_epochs"]),
            )
            count_curves.append(count_curve)

        pooled = _pool_count_curves(count_curves)
        if pooled.empty:
            continue

        tp = pooled["tp"].to_numpy(dtype=float)
        fp = pooled["fp"].to_numpy(dtype=float)
        fn = pooled["fn"].to_numpy(dtype=float)

        pooled["pooled_f1"] = binary_f1_from_counts(tp, fp, fn)
        curves.append(pooled[["step", "pooled_f1"]])

    return aggregate_seed_curves(curves, "pooled_f1")


def get_group_pooled_task_f1_fedavg(
    *,
    fedavg_items,
    task,
    graph_ids,
    split="val",
):
    target_graph_ids = [str(g) for g in graph_ids]
    curves = []

    for item in fedavg_items:
        count_curves = []
        fed_df = item["df"]
        local_epochs = int(item["local_epochs"])

        for gid in target_graph_ids:
            count_curve = get_task_count_curve(
                fed_df,
                phase="global_val_client_task",
                split=split,
                task=task,
                graph_id=gid,
                local_epochs=local_epochs,
            )
            count_curves.append(count_curve)

        pooled = _pool_count_curves(count_curves)
        if pooled.empty:
            continue

        tp = pooled["tp"].to_numpy(dtype=float)
        fp = pooled["fp"].to_numpy(dtype=float)
        fn = pooled["fn"].to_numpy(dtype=float)

        pooled["pooled_f1"] = binary_f1_from_counts(tp, fp, fn)
        curves.append(pooled[["step", "pooled_f1"]])

    return aggregate_seed_curves(curves, "pooled_f1")

## 4. Figure builders


In [63]:
def plot_validation_loss_panel(ax, *, local_items, fedavg_items, graph_ids, use_global_mean_post=False):
    local_agg = get_group_scalar_stats(
        items=local_items,
        phase="val_epoch",
        split="val",
        metric_col="eval_loss",
        graph_ids=graph_ids,
        use_graph_filter=False,
    )
    pre_agg = get_group_scalar_stats(
        items=fedavg_items,
        phase="val_epoch",
        split="val",
        metric_col="eval_loss",
        graph_ids=graph_ids,
        use_graph_filter=True,
    )
    if use_global_mean_post:
        post_agg = get_group_scalar_stats(
            items=fedavg_items,
            phase="global_val_mean",
            split="val",
            metric_col="eval_loss",
            graph_ids=None,
            use_graph_filter=False,
        )
    else:
        post_agg = get_group_scalar_stats(
            items=fedavg_items,
            phase="global_val_client",
            split="val",
            metric_col="eval_loss",
            graph_ids=graph_ids,
            use_graph_filter=True,
        )

    ok = False
    ok |= plot_mean_std(ax, local_agg, label="local standalone val")
    ok |= plot_mean_std(ax, pre_agg, label="fedavg client-local val (pre-aggregation)")
    ok |= plot_mean_std(ax, post_agg, label="fedavg global val (post-aggregation)")

    ax.set_title("validation loss comparison")
    ax.set_xlabel("communication round / local epoch")
    ax.set_ylabel("eval_loss")
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8)
    else:
        annotate_no_data(ax)


def plot_group_positive_f1_panel(ax, *, local_items, fedavg_items, graph_ids, source):
    phase_map = {
        "local": "val_epoch_task",
        "preagg": "val_epoch_task",
        "postagg": "global_val_client_task",
    }
    label_map = {
        "local": "local standalone on client (positive F1)",
        "preagg": "FedAvg client-local on client (pre-aggregation, positive F1)",
        "postagg": "FedAvg global on client (post-aggregation, positive F1)",
    }

    ok = False
    for task in TASKS:
        if source == "local":
            agg = get_group_task_stats(
                items=local_items,
                phase=phase_map[source],
                split="val",
                task=task,
                metric_col="positive_f1",
                graph_ids=graph_ids,
                use_graph_filter=False,
            )
        else:
            agg = get_group_task_stats(
                items=fedavg_items,
                phase=phase_map[source],
                split="val",
                task=task,
                metric_col="positive_f1",
                graph_ids=graph_ids,
                use_graph_filter=True,
            )
        ok |= plot_mean_std(ax, agg, label=task)

    ax.set_title(label_map[source])
    ax.set_xlabel("communication round / local epoch")
    ax.set_ylabel("positive_f1")
    ax.set_ylim(0.0, 1.05)
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)


def plot_group_delta_panel(ax, *, local_items, fedavg_items, graph_ids):
    ok = False
    for task in TASKS:
        agg = get_group_delta_stats(
            local_items=local_items,
            fedavg_items=fedavg_items,
            split="val",
            metric_col="positive_f1",
            graph_ids=graph_ids,
            task=task,
        )
        ok |= plot_mean_std(ax, agg, label=task)

    ax.axhline(0.0, linestyle="--", linewidth=1, color="black")
    ax.set_title("delta = local standalone - FedAvg global")
    ax.set_xlabel("communication round / local epoch")
    ax.set_ylabel("positive_f1 delta")
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)


def plot_group_pooled_panel(ax, *, local_items, fedavg_items, graph_ids, source):
    ok = False
    for task in TASKS:
        if source == "local":
            agg = get_group_pooled_task_f1_local(
                local_items=local_items,
                task=task,
                graph_ids=graph_ids,
            )
            title = "pooled LOCAL per-task F1"
        else:
            agg = get_group_pooled_task_f1_fedavg(
                fedavg_items=fedavg_items,
                task=task,
                graph_ids=graph_ids,
            )
            title = "pooled FedAvg GLOBAL per-task F1"

        ok |= plot_mean_std(ax, agg, label=task)

    ax.set_title(title)
    ax.set_xlabel("communication round / local epoch")
    ax.set_ylabel("pooled per-task F1")
    ax.set_ylim(0.0, 1.05)
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)

In [64]:
def make_global_overview_figure(row, exp_log):
    subset_clients = row["subset_clients"]
    model_tag = row["model_tag"]
    graph_ids = [str(g) for g in row["graph_ids"]]

    local_rows = filter_local_manifest(exp_log, subset_clients, model_tag, graph_ids)
    fedavg_rows = filter_fedavg_manifest(exp_log, subset_clients, model_tag)

    local_items = load_local_items(local_rows)
    fedavg_items = load_fedavg_items(fedavg_rows)

    fig = plt.figure(figsize=(16, 20))
    gs = GridSpec(4, 2, figure=fig)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])
    ax5 = fig.add_subplot(gs[2, 0])
    ax6 = fig.add_subplot(gs[2, 1])
    ax7 = fig.add_subplot(gs[3, 0])
    ax8 = fig.add_subplot(gs[3, 1])

    plot_validation_loss_panel(
        ax1,
        local_items=local_items,
        fedavg_items=fedavg_items,
        graph_ids=graph_ids,
        use_global_mean_post=True,
    )
    plot_group_positive_f1_panel(ax2, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids, source="local")
    plot_group_positive_f1_panel(ax3, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids, source="preagg")
    plot_group_positive_f1_panel(ax4, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids, source="postagg")
    plot_group_delta_panel(ax5, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids)
    plot_group_pooled_panel(ax6, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids, source="local")
    plot_group_pooled_panel(ax7, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids, source="fedavg")

    ax8.axis("off")
    meta_lines = [
        f"subset_clients = {subset_clients}",
        f"subset_size = {row['subset_size']}",
        f"n_seeds = {len(row['seeds'])}",
        f"model = {model_tag}",
        f"family_counts = {row['family_counts']}",
    ]
    ax8.text(0.01, 0.98, "\n".join(meta_lines), va="top", ha="left", fontsize=11)

    fig.suptitle(build_global_title(row), fontsize=16, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    return fig


def make_family_overview_figure(row, exp_log):
    subset_clients = row["subset_clients"]
    model_tag = row["model_tag"]
    graph_ids = [str(g) for g in row["graph_ids"]]

    local_rows = filter_local_manifest(exp_log, subset_clients, model_tag, graph_ids)
    fedavg_rows = filter_fedavg_manifest(exp_log, subset_clients, model_tag)

    local_items = load_local_items(local_rows)
    fedavg_items = load_fedavg_items(fedavg_rows)

    fig = plt.figure(figsize=(16, 20))
    gs = GridSpec(4, 2, figure=fig)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])
    ax5 = fig.add_subplot(gs[2, 0])
    ax6 = fig.add_subplot(gs[2, 1])
    ax7 = fig.add_subplot(gs[3, 0])
    ax8 = fig.add_subplot(gs[3, 1])

    plot_validation_loss_panel(
        ax1,
        local_items=local_items,
        fedavg_items=fedavg_items,
        graph_ids=graph_ids,
        use_global_mean_post=False,
    )
    plot_group_positive_f1_panel(ax2, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids, source="local")
    plot_group_positive_f1_panel(ax3, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids, source="preagg")
    plot_group_positive_f1_panel(ax4, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids, source="postagg")
    plot_group_delta_panel(ax5, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids)
    plot_group_pooled_panel(ax6, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids, source="local")
    plot_group_pooled_panel(ax7, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids, source="fedavg")

    ax8.axis("off")
    meta_lines = [
        f"family = {row['family']}",
        f"graph_ids = {graph_ids}",
        f"subset_size = {row['subset_size']}",
        f"n_seeds = {len(row['seeds'])}",
        f"model = {model_tag}",
    ]
    ax8.text(0.01, 0.98, "\n".join(meta_lines), va="top", ha="left", fontsize=11)

    fig.suptitle(build_family_title(row), fontsize=16, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    return fig


def make_client_overview_figure(row, exp_log):
    subset_clients = row["subset_clients"]
    model_tag = row["model_tag"]
    graph_id = str(row["graph_id"])
    graph_ids = [graph_id]

    local_rows = filter_local_manifest(exp_log, subset_clients, model_tag, graph_ids)
    fedavg_rows = filter_fedavg_manifest(exp_log, subset_clients, model_tag)

    local_items = load_local_items(local_rows)
    fedavg_items = load_fedavg_items(fedavg_rows)

    fig = plt.figure(figsize=(16, 15))
    gs = GridSpec(3, 2, figure=fig)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])
    ax5 = fig.add_subplot(gs[2, 0])
    ax6 = fig.add_subplot(gs[2, 1])

    plot_validation_loss_panel(
        ax1,
        local_items=local_items,
        fedavg_items=fedavg_items,
        graph_ids=graph_ids,
        use_global_mean_post=False,
    )
    plot_group_positive_f1_panel(ax2, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids, source="local")
    plot_group_positive_f1_panel(ax3, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids, source="preagg")
    plot_group_positive_f1_panel(ax4, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids, source="postagg")
    plot_group_delta_panel(ax5, local_items=local_items, fedavg_items=fedavg_items, graph_ids=graph_ids)

    ax6.axis("off")
    meta_lines = [
        f"family = {row['family']}",
        f"graph_id = {graph_id}",
        f"n_seeds = {len(row['seeds'])}",
        f"model = {model_tag}",
    ]
    ax6.text(0.01, 0.98, "\n".join(meta_lines), va="top", ha="left", fontsize=11)

    fig.suptitle(build_client_title(row), fontsize=16, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    return fig

## 5. PDF export and one-run runner


In [65]:
def build_global_pdf(global_run_table, exp_log, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with PdfPages(out_path) as pdf:
        for _, row in global_run_table.iterrows():
            fig = make_global_overview_figure(row, exp_log)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

    print(f"saved -> {out_path.resolve()}")


def build_family_pdf(family_run_table, exp_log, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with PdfPages(out_path) as pdf:
        for _, row in family_run_table.iterrows():
            fig = make_family_overview_figure(row, exp_log)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

    print(f"saved -> {out_path.resolve()}")


def build_client_pdf(client_run_table, exp_log, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with PdfPages(out_path) as pdf:
        for _, row in client_run_table.iterrows():
            fig = make_client_overview_figure(row, exp_log)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

    print(f"saved -> {out_path.resolve()}")

In [66]:
global_run_table = build_global_run_table(selected_subset, exp_log)
family_run_table = build_family_run_table(selected_subset, exp_log)
client_run_table = build_client_run_table(selected_subset, exp_log)

OUT_DIR = Path("./plot_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

build_global_pdf(
    global_run_table,
    exp_log,
    OUT_DIR / "subset_global_overview.pdf",
)

build_family_pdf(
    family_run_table,
    exp_log,
    OUT_DIR / "subset_family_overview.pdf",
)

build_client_pdf(
    client_run_table,
    exp_log,
    OUT_DIR / "subset_client_overview.pdf",
)

global_run_table: 1
family_run_table: 5
client_run_table: 20


,subset_id,subset_clients,subset_size,graph_ids,family_order,family_to_graph_ids,family_counts,model_tag,local_epochs,seeds
0,combined_five_family_benchmark_87_157_149_109_...,87|157|149|109|143|114|117|2|88|145|166|141|65...,20,"[87, 157, 149, 109, 143, 114, 117, 2, 88, 145,...","[strong_decreasing, mild_decreasing, flat_bala...","{'strong_decreasing': ['65', '6', '131', '119'...","{'strong_decreasing': 4, 'mild_decreasing': 4,...",mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...,1,"[0, 1, 2]"


,subset_id,subset_clients,family,subset_size,graph_ids,model_tag,local_epochs,seeds
0,combined_five_family_benchmark_87_157_149_109_...,87|157|149|109|143|114|117|2|88|145|166|141|65...,strong_decreasing,4,"[65, 6, 131, 119]",mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...,1,"[0, 1, 2]"
1,combined_five_family_benchmark_87_157_149_109_...,87|157|149|109|143|114|117|2|88|145|166|141|65...,mild_decreasing,4,"[143, 114, 117, 2]",mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...,1,"[0, 1, 2]"
2,combined_five_family_benchmark_87_157_149_109_...,87|157|149|109|143|114|117|2|88|145|166|141|65...,flat_balanced,4,"[87, 157, 149, 109]",mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...,1,"[0, 1, 2]"
3,combined_five_family_benchmark_87_157_149_109_...,87|157|149|109|143|114|117|2|88|145|166|141|65...,mild_increasing,4,"[88, 145, 166, 141]",mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...,1,"[0, 1, 2]"
4,combined_five_family_benchmark_87_157_149_109_...,87|157|149|109|143|114|117|2|88|145|166|141|65...,strong_increasing,4,"[96, 162, 107, 0]",mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...,1,"[0, 1, 2]"


,subset_id,subset_clients,family,graph_id,model_tag,local_epochs,seeds
0,combined_five_family_benchmark_87_157_149_109_...,87|157|149|109|143|114|117|2|88|145|166|141|65...,flat_balanced,87,mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...,1,"[0, 1, 2]"
1,combined_five_family_benchmark_87_157_149_109_...,87|157|149|109|143|114|117|2|88|145|166|141|65...,flat_balanced,157,mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...,1,"[0, 1, 2]"
2,combined_five_family_benchmark_87_157_149_109_...,87|157|149|109|143|114|117|2|88|145|166|141|65...,flat_balanced,149,mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...,1,"[0, 1, 2]"
3,combined_five_family_benchmark_87_157_149_109_...,87|157|149|109|143|114|117|2|88|145|166|141|65...,flat_balanced,109,mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...,1,"[0, 1, 2]"
4,combined_five_family_benchmark_87_157_149_109_...,87|157|149|109|143|114|117|2|88|145|166|141|65...,mild_decreasing,143,mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...,1,"[0, 1, 2]"


saved -> /Users/andreali/Documents/Subgraph_Federated_Learning/andrea/plot_outputs/subset_global_overview.pdf
saved -> /Users/andreali/Documents/Subgraph_Federated_Learning/andrea/plot_outputs/subset_family_overview.pdf
saved -> /Users/andreali/Documents/Subgraph_Federated_Learning/andrea/plot_outputs/subset_client_overview.pdf
